# Credit Risk Modeling - German Credit

**Tabular Classification** — Predict creditworthiness from the German Credit dataset.

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Dataset: German Credit (UCI OpenML, 1000 rows × 21 columns)
Target: `class` (good/bad credit)

## 2 · Project Overview

The German Credit dataset is a classic benchmark from UCI for credit risk assessment. We predict whether an applicant has **good** or **bad** credit risk based on financial and personal attributes. With only 1000 rows, this tests model performance on small datasets.

## 3 · Learning Objectives

1. Work with the classic German Credit dataset from UCI.
2. Handle ordinal and nominal categorical features properly.
3. Benchmark small-dataset performance across models.
4. Understand credit scoring fundamentals.
5. Compare automated ML tools on a small dataset.

## 4 · Problem Statement

Given an applicant's **financial status**, **employment**, **housing**, **credit history**, and **loan purpose**, classify them as **good credit** or **bad credit** risk.

## 5 · Why This Project Matters

- One of the oldest and most-studied credit risk datasets in ML.
- Small size (1000 rows) makes it ideal for comparing model robustness.
- Directly applicable to real banking credit decisions.
- Teaches working with mixed feature types on limited data.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 1,000 |
| **Columns** | 21 |
| **Source** | sklearn `fetch_openml(data_id=31)` |
| **Target** | `class` (good / bad) |
| **Imbalance** | ~70% good, 30% bad |

## 7 · Dataset Source and License Notes

- **Source**: UCI Machine Learning Repository, accessed via OpenML.
- **Original**: Prof. Dr. Hans Hofmann, Universität Hamburg.
- **License**: Public domain for research.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [("catboost", None), ("lightgbm", None), ("xgboost", None),
                 ("flaml", None), ("lazypredict", None), ("kagglehub", None)]:
    _install_if_missing(pkg, imp)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, re, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, average_precision_score)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("Imports done.")

Imports done.


## 10 · Configuration

In [3]:
SAVE_DIR = r"E:/Github/Machine-Learning-Projects/Classification/Credit Risk Modeling - German Credit"
os.makedirs(SAVE_DIR, exist_ok=True)
TARGET = "class"
print(f"Save dir: {SAVE_DIR}")

Save dir: E:/Github/Machine-Learning-Projects/Classification/Credit Risk Modeling - German Credit


## 11 · Dataset Download and Loading

In [4]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=31, target_column="class", as_frame=True, parser="auto")
df = data.frame

# Convert category columns to numeric codes
for col in df.select_dtypes(["category"]).columns:
    df[col] = df[col].cat.codes

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df[TARGET].value_counts()}")
df.head()

Shape: (1000, 21)
Target distribution:
class
1    700
0    300
Name: count, dtype: int64


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,1,6,1,7,1169,4,3,4,3,2,...,2,67,1,1,2,3,1,1,1,1
1,0,48,3,7,5951,2,0,2,0,2,...,2,22,1,1,1,3,1,0,1,0
2,3,12,1,4,2096,2,1,2,3,2,...,2,49,1,1,1,2,2,0,1,1
3,1,42,3,5,7882,2,1,2,3,1,...,0,45,1,0,1,3,2,0,1,1
4,1,24,2,1,4870,2,0,3,3,2,...,1,53,1,0,2,3,2,0,1,0


## 12 · Data Validation Checks

In [5]:
print(f"Missing values:\n{df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"Data types:\n{df.dtypes.value_counts()}")
df.describe()

Missing values:
0
Duplicates: 0
Data types:
int8     14
int64     7
Name: count, dtype: int64


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
count,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,...,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,1.582000,20.903000,2.219000,4.12500,3271.258000,2.1450,1.525000,2.973000,1.878000,1.866000,...,1.714000,35.546000,0.908000,1.071000,1.407000,2.312000,1.155000,0.404000,0.963000,0.700000
std,1.253334,12.058814,1.064035,2.47457,2822.736876,1.1114,1.344315,1.118715,1.350904,0.445244,...,1.154789,11.375469,0.421561,0.531264,0.577654,1.071356,0.362086,0.490943,0.188856,0.458487
min,0.000000,4.000000,0.000000,0.00000,250.000000,0.0000,0.000000,1.000000,0.000000,0.000000,...,0.000000,19.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,0.000000,12.000000,1.000000,2.00000,1365.500000,2.0000,0.000000,2.000000,0.000000,2.000000,...,1.000000,27.000000,1.000000,1.000000,1.000000,2.000000,1.000000,0.000000,1.000000,0.000000
50%,1.000000,18.000000,3.000000,5.00000,2319.500000,2.0000,1.000000,3.000000,3.000000,2.000000,...,2.000000,33.000000,1.000000,1.000000,1.000000,3.000000,1.000000,0.000000,1.000000,1.000000
75%,3.000000,24.000000,3.000000,7.00000,3972.250000,2.0000,3.000000,4.000000,3.000000,2.000000,...,3.000000,42.000000,1.000000,1.000000,2.000000,3.000000,1.000000,1.000000,1.000000,1.000000
max,3.000000,72.000000,4.000000,9.00000,18424.000000,4.0000,4.000000,4.000000,3.000000,2.000000,...,3.000000,75.000000,2.000000,2.000000,4.000000,3.000000,2.000000,1.000000,1.000000,1.000000


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [c for c in num_cols if c != TARGET][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=30, ax=ax, edgecolor="black")
    ax.set_title(col)
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_distributions.png"), dpi=100)
plt.show()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_correlation.png"), dpi=100)
plt.show()

## 14 · Target Analysis

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "coral"])
ax.set_title("Credit Risk Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_target.png"), dpi=100)
plt.show()
print(f"Class balance: {df[TARGET].value_counts(normalize=True).to_dict()}")

Class balance: {1: 0.7, 0: 0.3}


## 15 · Train / Test Split

In [8]:
# Encode target if needed
if df[TARGET].dtype == object:
    le_t = LabelEncoder()
    df[TARGET] = le_t.fit_transform(df[TARGET])

y = df[TARGET]
X = df.drop(columns=[TARGET])

# Sanitize column names
X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", str(c)) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target: {y_train.value_counts().to_dict()}")

Train: (800, 20), Test: (200, 20)
Train target: {1: 560, 0: 240}


## 16 · Preprocessing Strategy

All features are now numeric (category codes from OpenML). No missing values. Small dataset — no sampling needed.

In [9]:
print(f"Missing: {X_train.isnull().sum().sum()}")
print(f"All numeric: {X_train.select_dtypes(include=[np.number]).shape[1]} / {X_train.shape[1]}")

Missing: 0
All numeric: 20 / 20


## 17 · Feature Engineering

In [10]:
# With only 1000 rows, we keep features simple to avoid overfitting
print(f"Features: {X_train.shape[1]} (no extra engineering on small dataset)")

Features: 20 (no extra engineering on small dataset)


## 18 · Baseline Model

In [11]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_proba = baseline.predict_proba(X_test)[:, 1]

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_f1 = f1_score(y_test, baseline_pred)
baseline_auc = roc_auc_score(y_test, baseline_proba)
print(f"Baseline Logistic Regression:")
print(f"  Accuracy: {baseline_acc:.4f}  F1: {baseline_f1:.4f}  ROC-AUC: {baseline_auc:.4f}")
print(classification_report(y_test, baseline_pred))

Baseline Logistic Regression:
  Accuracy: 0.6500  F1: 0.7177  ROC-AUC: 0.7358
              precision    recall  f1-score   support

           0       0.45      0.68      0.54        60
           1       0.82      0.64      0.72       140

    accuracy                           0.65       200
   macro avg       0.63      0.66      0.63       200
weighted avg       0.71      0.65      0.66       200



## 19 · LazyPredict Benchmark

In [12]:
# LazyPredict — quick benchmark of many classifiers
from lazypredict.Supervised import LazyClassifier

lp_clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None)
lp_models, lp_predictions = lp_clf.fit(X_train, X_test, y_train, y_test)

print("\nLazyPredict Results (top 15):")
print(lp_models.head(15).to_string())


LazyPredict Results (top 15):
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
AdaBoostClassifier                0.770           0.707143  0.771429  0.764980   0.762682   0.770    0.087814
BaggingClassifier                 0.740           0.690476  0.743155  0.740000   0.740000   0.740    0.046835
QuadraticDiscriminantAnalysis     0.705           0.679762  0.752619  0.712236   0.725283   0.705    0.018754
RandomForestClassifier            0.765           0.675000  0.776905  0.750363   0.752278   0.765    0.184934
ExtraTreesClassifier              0.765           0.670238  0.791012  0.748273   0.752174   0.765    0.151083
NearestCentroid                   0.655           0.663095  0.725357  0.668938   0.713146   0.655    0.018104
CatBoostClassifier                0.760           0.661905  0.785357  0.741803   0.746134

## 20 · FLAML AutoML

In [13]:
# FLAML AutoML
try:
    from flaml import AutoML
    flaml_clf = AutoML()
    flaml_clf.fit(X_train, y_train, task="classification", time_budget=60,
                  metric="f1", verbose=0, seed=SEED)
    flaml_pred = flaml_clf.predict(X_test)
    flaml_proba = flaml_clf.predict_proba(X_test)[:, 1] if hasattr(flaml_clf, "predict_proba") else None
    print(f"FLAML best model: {flaml_clf.best_estimator}")
    print(f"FLAML F1: {f1_score(y_test, flaml_pred):.4f}")
    print(f"FLAML Accuracy: {accuracy_score(y_test, flaml_pred):.4f}")
    if flaml_proba is not None:
        print(f"FLAML ROC-AUC: {roc_auc_score(y_test, flaml_proba):.4f}")
    flaml_ok = True
except Exception as e:
    print(f"FLAML failed: {e}")
    flaml_ok = False

FLAML failed: XGBClassifier.fit() got an unexpected keyword argument 'callbacks'


## 21 · Top 3 Models — CatBoost, LightGBM, XGBoost

In [14]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import torch
device_cat = "GPU" if torch.cuda.is_available() else "CPU"
device_lgb = "gpu" if torch.cuda.is_available() else "cpu"
device_xgb = "cuda" if torch.cuda.is_available() else "cpu"

models = {
    "CatBoost": CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, verbose=0,
        task_type=device_cat, random_seed=SEED, eval_metric="F1",
        early_stopping_rounds=50
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_lgb, random_state=SEED, verbose=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        device=device_xgb, random_state=SEED, eval_metric="logloss",
        early_stopping_rounds=50
    ),
}

results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    t0 = time.time()
    if name == "XGBoost":
        model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    elif name == "CatBoost":
        model.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)
    else:
        model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, proba)

    results[name] = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "time": elapsed}
    print(f"  Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  ROC-AUC: {auc:.4f}  ({elapsed:.1f}s)")
    print(classification_report(y_test, preds))

results_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\n=== Top 3 Model Comparison ===")
print(results_df.to_string())


Training CatBoost...


  Accuracy: 0.7900  Precision: 0.8025  Recall: 0.9286  F1: 0.8609  ROC-AUC: 0.7827  (3.0s)
              precision    recall  f1-score   support

           0       0.74      0.47      0.57        60
           1       0.80      0.93      0.86       140

    accuracy                           0.79       200
   macro avg       0.77      0.70      0.72       200
weighted avg       0.78      0.79      0.77       200


Training LightGBM...


  Accuracy: 0.7150  Precision: 0.7785  Recall: 0.8286  F1: 0.8028  ROC-AUC: 0.7231  (1.1s)
              precision    recall  f1-score   support

           0       0.53      0.45      0.49        60
           1       0.78      0.83      0.80       140

    accuracy                           0.71       200
   macro avg       0.65      0.64      0.64       200
weighted avg       0.70      0.71      0.71       200


Training XGBoost...


  Accuracy: 0.7450  Precision: 0.7730  Recall: 0.9000  F1: 0.8317  ROC-AUC: 0.7823  (0.4s)
              precision    recall  f1-score   support

           0       0.62      0.38      0.47        60
           1       0.77      0.90      0.83       140

    accuracy                           0.74       200
   macro avg       0.70      0.64      0.65       200
weighted avg       0.73      0.74      0.72       200


=== Top 3 Model Comparison ===
          accuracy  precision    recall        f1   roc_auc      time
CatBoost     0.790   0.802469  0.928571  0.860927  0.782738  2.977484
XGBoost      0.745   0.773006  0.900000  0.831683  0.782262  0.361532
LightGBM     0.715   0.778523  0.828571  0.802768  0.723095  1.135557


## 22 · Top 3 Model Selection

In [15]:
print("=== Final Ranking ===")
print(results_df[["f1", "roc_auc", "accuracy"]].to_string())

=== Final Ranking ===
                f1   roc_auc  accuracy
CatBoost  0.860927  0.782738     0.790
XGBoost   0.831683  0.782262     0.745
LightGBM  0.802768  0.723095     0.715


## 23 · Final Evaluation

In [16]:
print("Comparison vs baseline:")
print(f"  Baseline F1: {baseline_f1:.4f} | ROC-AUC: {baseline_auc:.4f}")
for name, vals in results.items():
    print(f"  {name} F1: {vals['f1']:.4f} | ROC-AUC: {vals['roc_auc']:.4f}")

Comparison vs baseline:
  Baseline F1: 0.7177 | ROC-AUC: 0.7358
  CatBoost F1: 0.8609 | ROC-AUC: 0.7827
  LightGBM F1: 0.8028 | ROC-AUC: 0.7231
  XGBoost F1: 0.8317 | ROC-AUC: 0.7823


## 24 · Error Analysis

In [17]:
# Error Analysis — examine misclassifications from best model
best_name = results_df.index[0]
best_model = models[best_name]
best_preds = best_model.predict(X_test)

errors = X_test.copy()
errors["y_true"] = y_test.values
errors["y_pred"] = best_preds
errors["correct"] = errors["y_true"] == errors["y_pred"]

print(f"Best model: {best_name}")
print(f"Total test samples: {len(errors)}")
print(f"Correct: {errors['correct'].sum()} | Wrong: {(~errors['correct']).sum()}")
print(f"Error rate: {(~errors['correct']).mean():.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, best_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()
print("Confusion matrix saved.")

Best model: CatBoost
Total test samples: 200
Correct: 158 | Wrong: 42
Error rate: 0.2100
Confusion matrix saved.


## 25 · Interpretation and Business Insight

- With only 1000 rows, simpler models often compete with complex ones.
- Credit history and account status features are typically most predictive.
- The 70/30 class split requires attention to the minority class.
- This dataset shows that more data is often more valuable than more complex models.

## 26 · Limitations

- Very small dataset (1000 rows) — high variance in performance estimates.
- German-specific features may not generalize internationally.
- Data from the 1990s — credit patterns have changed significantly.
- Encoded categories lose interpretability.

## 27 · How to Improve

1. Use cross-validation (5-fold or 10-fold) instead of single train/test split.
2. Try TabPFN-v2 which is designed for small tabular datasets.
3. Apply cost-sensitive learning (misclassifying bad as good costs more).
4. Use SHAP for feature importance analysis.

## 28 · Production Considerations

- Regulatory compliance (German Banking Act, EU GDPR).
- Need for explainability in credit decisions.
- Small training data means high uncertainty — use prediction intervals.
- Regular model retraining as economic conditions change.

## 29 · Common Mistakes

1. Overfitting on 1000 rows with complex models.
2. Not using stratified splits on imbalanced data.
3. Ignoring the cost asymmetry between false positives and false negatives.
4. Over-engineering features on small datasets.

## 30 · Mini Challenge

1. Run 10-fold cross-validation and report mean ± std for each model.
2. Try TabPFN-v2 and compare it to boosting models on this small dataset.
3. Create a cost matrix and optimize for minimum total cost.
4. Compare results with and without feature engineering.

## 31 · Final Summary

- The German Credit dataset is a foundational benchmark for credit risk ML.
- Small datasets challenge even advanced models — simplicity often wins.
- Cross-validation is essential when data is limited.
- Business context (cost of errors) must drive model selection.

In [18]:
# Save metrics to JSON
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in results.items()}
metrics["baseline"] = {"accuracy": round(baseline_acc, 4), "f1": round(baseline_f1, 4)}
with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {os.path.join(SAVE_DIR, 'metrics.json')}")

Metrics saved to E:/Github/Machine-Learning-Projects/Classification/Credit Risk Modeling - German Credit\metrics.json
